In [10]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor

# load environment variables
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

In [11]:
# test search tool
MAX_SEARCH_RESULTS = 2
search_tool = TavilySearchResults(max_results=MAX_SEARCH_RESULTS)
search_result = search_tool.invoke("What is the capital of France?")
search_result

[{'url': 'https://en.wikipedia.org/wiki/Paris',
  'content': 'As the capital of France, Paris is the seat of France\'s national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. The President of the French Republic resides at the Élysée Palace. The Prime Minister\'s seat is at the Hôtel Matignon. Government ministries are located in various parts of the city, many near the Hôtel Matignon. [...] Edit links\n\n Article\n Talk\n\n Read\n View source\n View history\n\nTools\n\nActions\n\n Read\n View source\n View history\n\nGeneral\n\n What links here\n Related changes\n Upload file\n Permanent link\n Page information\n Cite this page\n Get shortened URL\n Switch to legacy parser\n\nPrint/export\n\n Download as PDF\n Printable version\n\nIn other projects\n\n Abstract Wikipedia\n Wikimedia Commons\n Wikinews\n Wikiquote\n Wikivoyage\n Wikidata item\n\nAppearance\n\nCoordinates: 48°51′24″N 2°21′08″E\ufeff / \uf

In [12]:
# Define the Function / Tool
@tool
def get_current_temperature(location):
    """
    Fetch real-time temperature for a given location using OpenWeatherMap API.
    """
    url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid={OPENWEATHER_API_KEY}&units=metric"
    response = requests.get(url)
    data = response.json()

    if response.status_code == 200:
        return data['main']['temp']
    else:
        raise Exception(f"Error fetching weather: {data.get('message', 'Unknown error')}")
    
# sample usage
get_current_temperature("London")

26.48

In [13]:
# tool to fetch weather information
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """
    
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n",
        f"Temperature: {data['current']['temperature']}°C\n",
        f"Weather: {data['current']['weather_descriptions']}°C\n",
        f"Humidity: {data['current']['humidity']}°C\n",
    )

# sample usage
get_weather_data("London")

('City: London\n',
 'Temperature: 24°C\n',
 "Weather: ['Partly cloudy']°C\n",
 'Humidity: 57°C\n')

In [21]:
# initialize LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# test llm
llm_result = llm.invoke("What year is it?")
llm_result.content

'It is currently 2022.'

In [16]:
# pre-defined prompts
prompt = hub.pull("hwchase17/react")
prompt

c:\Users\Utkarsh\anaconda3\envs\langchain_agent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
# configure tools
tools = [search_tool, get_weather_data]
tools

[TavilySearchResults(max_results=2),
 StructuredTool(name='get_weather_data', description='get_weather_data(city: str) -> str - Fetch current weather information for a city.', args_schema=<class 'pydantic.v1.main.get_weather_dataSchema'>, func=<function get_weather_data at 0x000002345E5EDBC0>)]

In [18]:
# CREATE AGENT
agent = create_react_agent(
    llm = llm,
    tools = tools,
    prompt = prompt
)

# AGENT EXECUTOR
agent_executor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True
)

In [19]:
# RUN AGENT
USER_QUERY = "Find the capital of India and its current weather"
response = agent_executor.invoke({
    "input": (
        USER_QUERY
    )
})

print(response)



> Entering new AgentExecutor chain...
I need to find the capital of India and then get the current weather information for that city.
Action: tavily_search_results_json
Action Input: "capital of India"[{'url': 'https://www.burningcompass.com/countries/india/india-capital.html', 'content': '| Category | Specific Fact | Details / Value |\n --- \n| Official Name | Capital City | New Delhi (planned capital since 1911) |\n| Country | India | National capital; seat of President, Parliament, Supreme Court |\n| Administered by | New Delhi Municipal Council (NDMC) | Covers Lutyens\' Delhi; 80% Central Govt influence, 20% Delhi Govt |\n| Established | Foundation stone laid | 12 December 1911 (Delhi Durbar by King George V) |\n| Inaugurated | Official opening | 13 February 1931 |\n| Architects | Primary planners | Sir Edwin Lutyens & Sir Herbert Baker |\n| Motto | Official | Śrama ēva jayatē ("Labour is victory") |\n| Coordinates | Central point | 28°36′50″N 77°12′32″E (28.6139°N 77.2089°E) |\n

In [20]:
sdfsdfsdfsd

NameError: name 'sdfsdfsdfsd' is not defined

In [ ]:
# RUN AGENT
USER_QUERY = "Find the capital of India"
response = agent_executor.invoke({
    "input": (
        USER_QUERY
    )
})

print(response)



> Entering new AgentExecutor chain...
I should use the search engine to quickly find the answer
Action: tavily_search_results_json
Action Input: Capital of India[{'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': '## Modern period\n\n[edit]\n\n In 1858, Allahabad the capital of India for a day when it also served as the capital of North-Western Provinces.\n During the British Raj, until 1911, Calcutta was the capital of India.\n By the latter half of the 19th century, Shimla had become the summer capital.\n King George V proclaimed the transfer of the capital from Calcutta to Delhi at the climax of the 1911 Delhi Durbar on 12 December 1911. The buildings housing the Viceroy, government, and parliament were inaugurated in early 1931.\n\n## References\n\n[edit] [...] Appearance\n\nFrom Wikipedia, the free encyclopedia\n\n|  |\n\n| This article needs more citations. Please help improve this article by adding citations to reliable sources. Unsourced material ma

In [ ]:
# FINAL OUTPUT
print(response["output"])

New Delhi


In [ ]:
# RUN AGENT
USER_QUERY = "Tell me the latest news on Iran"
response = agent_executor.invoke({
    "input": (
        USER_QUERY
    )
})

print(response['output'])



> Entering new AgentExecutor chain...
I should use tavily_search_results_json to get the most up-to-date information on Iran.
Action: tavily_search_results_json
Action Input: Iran latest news[{'url': 'https://www.nbcnews.com/iran', 'content': 'IE 11 is not supported. For an optimal experience visit our site on another browser.\n\n# Iran\n\nTop stories from Iran.\n\n Middle East conflict\n Middle East news\n World news\n\n3h ago\n\n## Iran\n\n## Iran\n\n## Iran indefinitely closes Strait of Hormuz and attacks ship, prompting more U.S. strikes\n\nEarlier, Supreme Leader Mojtaba Khamenei pledged to take revenge for the death of his father.\n\n1d ago\n\n## Iran\n\n## Iran\n\n## Trump says U.S. agreed to Iran’s request to continue talks, but ceasefire is over\n\nDaily tanker traffic through the Strait of Hormuz appears to have slowed as attacks stoke concerns about the recovery of the global oil industry.\n\n2d ago\n\n## Iran\n\n## Iran\n\nFORSUBSCRIBERS\n\n## Iran’s supreme leader buried